In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix

# Verificamos la versión de TensorFlow (buena práctica en notebooks)
print(f"TensorFlow Version: {tf.__version__}")
sns.set_theme(style="white")

In [ ]:
# Cargar el dataset Fashion MNIST
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

# Etiquetas de las clases
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# Normalización: transformamos la matriz de enteros (0-255) a flotantes (0-1)
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Redimensionar para añadir el canal de color (escala de grises = 1)
# Forma esperada por la CNN: (batch_size, height, width, channels)
X_train = np.expand_dims(X_train, -1)
X_test = np.expand_dims(X_test, -1)

print(f"Forma del tensor de entrenamiento: {X_train.shape}")

In [ ]:
model = models.Sequential([
    # Bloque de Extracción de Características 1
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    
    # Bloque de Extracción de Características 2
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Clasificador Denso
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3), # Regularización
    layers.Dense(10, activation='softmax') # Capa de salida para clasificación multiclase
])

model.summary()

In [ ]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Configurar el callback
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Entrenamiento del modelo
history = model.fit(X_train, y_train, 
                    epochs=20, 
                    batch_size=64,
                    validation_split=0.2, 
                    callbacks=[early_stop])

In [ ]:
# Extraer datos del historial
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(len(acc))

plt.figure(figsize=(12, 5))

# Gráfica de Precisión
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

# Gráfica de Pérdida
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')

plt.tight_layout()
plt.show()

In [ ]:
# Evaluación final sobre el conjunto de prueba que el modelo nunca ha visto
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Precisión en el conjunto de prueba: {test_acc:.4f}")

# Predecir algunas imágenes y visualizarlas
y_pred_probs = model.predict(X_test)
y_pred_classes = np.argmax(y_pred_probs, axis=1)

# Visualizar 5 predicciones
plt.figure(figsize=(15, 3))
for i in range(5):
    plt.subplot(1, 5, i+1)
    plt.imshow(X_test[i].reshape(28, 28), cmap='gray')
    pred_label = class_names[y_pred_classes[i]]
    true_label = class_names[y_test[i]]
    color = 'blue' if pred_label == true_label else 'red'
    plt.title(f"Pred: {pred_label}\nReal: {true_label}", color=color)
    plt.axis('off')
plt.show()